In [21]:
pip install --upgrade langchain langchain-google-genai chromadb


Note: you may need to restart the kernel to use updated packages.Requirement already satisfied: langchain in c:\users\hp\appdata\local\programs\python\python312\lib\site-packages (1.3.14)




[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [22]:
%pip install --upgrade --quiet  langchain-community

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [23]:
pip install langchain-core langchain-text-splitters


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [24]:
pip install sentence-transformers


Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement sentence-transformers (from versions: none)

[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for sentence-transformers


In [25]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI


In [26]:
API_KEY = "YOUR_GOOGLE_API_KEY_HERE"

In [27]:
# 1. Document text (embedded in code)
doc_text = """
Elon Musk is a technology entrepreneur and engineer known for founding SpaceX and Tesla.
He was born on June 28, 1971, in Pretoria, South Africa.
His major achievements include advancing space exploration and electric vehicles.
Musk is also involved with Neuralink and The Boring Company.
This document provides a brief overview of Musk's background and accomplishments.
"""

document = Document(page_content=doc_text, metadata={"source": "in-memory-doc"})

In [28]:
# 2. Split into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs_split = text_splitter.split_documents([document])

In [29]:
# 3. Use SentenceTransformers embeddings via HuggingFaceEmbeddings wrapper
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

ImportError: Could not import sentence_transformers python package. Please install it with `pip install sentence-transformers`.

In [ ]:
# 4. Create FAISS vector store
vectorstore = FAISS.from_documents(docs_split, embeddings)


In [ ]:
# 5. Initialize Gemini 2.5 Flash LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.0,
    max_tokens=None,
    api_key=GOOGLE_API_KEY,
)

In [ ]:
# 6. Setup RetrievalQA chain
retriever = vectorstore.as_retriever()

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=True,
    chain_type="stuff",
)

In [ ]:
# 7. Query the system
query = "Who is Elon Musk and what are his major achievements?"
result = qa_chain.invoke({"query": query})

print("Answer:", result['result'])
print("\nSource Documents:")
for doc in result['source_documents']:
    print(f"- {doc.page_content}")

Answer: Elon Musk is a technology entrepreneur and engineer known for founding SpaceX and Tesla. His major achievements include advancing space exploration and electric vehicles.

Source Documents:
- Elon Musk is a technology entrepreneur and engineer known for founding SpaceX and Tesla.
He was born on June 28, 1971, in Pretoria, South Africa.
His major achievements include advancing space exploration and electric vehicles.
Musk is also involved with Neuralink and The Boring Company.
This document provides a brief overview of Musk's background and accomplishments.
